# Find closest sequence

Typically, ClinVar uses only one or a few reference sequences for a human gene.  We
should find the sequences from other species that are most similar to the human
reference sequence.  

To illustrate, let's consider ABCD1.  Below we see that there are 2,067 variants
in the ClinVar database. 

**Note:** I had to update the code for the `get_variant_details()` function.
The `@RecordType` appears to have changed to `classified`.  I suspect that this
is due to the expansion of the database to include both germline and somatic
mutations.  The new function needs more comprehensive testing but it seems to
work fine on the ABCD1 variants.

In [1]:
run kondrashov

In [2]:
gene = "ABCD1"

In [4]:
nvars, varids = get_variant_ids(gene)
nvars

2067

The following code confirms that every variant for this gene is
based on the [NM_000033.4](https://www.ncbi.nlm.nih.gov/nuccore/nm_000033.4)
reference transcript.  

This encodes the
[NNP_000024.2](https://www.ncbi.nlm.nih.gov/protein/7262393) reference protein.

**Note:** [`tqdm`](https://github.com/tqdm/tqdm) is a package that introduces a
progress bar for `for` loops.

In [37]:
from tqdm import tqdm

In [40]:
for varid in tqdm(varids):
    varrec = get_variant(varid)
    name, acc, mut, pmut, muttype, status = get_variant_details(tmp)
    if acc != "NM_000033.4":
        print(varid, acc)

100%|██████████| 1667/1667 [11:14<00:00,  2.47it/s]


# Human sequences

There are 3 unique human ABCD1 protein sequences in the `fasta` file.

In [20]:
hum_ids = ['NP_000024.2', 'NP_001427676.1', 'XP_047297873.1']

In [21]:
# Collect unique human sequences
print('{0} human sequences\n'.format(gene))
hum_sequences = []
hum_records = []
inc = 0
exc = 0
with Entrez.efetch(
    db="protein", rettype="gb", retmode="text", id=hum_ids,
) as handle:
    for seq_record in SeqIO.parse(handle, "gb"):
        sp = get_species(seq_record)
        seq = seq_record.seq
        if seq in hum_sequences:
            print("\t{0}\t({1} aa)\t{2}\t*** the same as sequence {3} (excluded) ***".format(seq_record.id, len(seq), seq_record.description, hum_sequences.index(seq)))
            exc += 1
        else:
            hum_sequences.append(seq)
            print("{3}:\t{0}\t({1} aa)\t{2}".format(seq_record.id, len(seq), seq_record.description, inc))
            hum_records.append(seq_record)
            inc += 1

ABCD1 human sequences

0:	NP_000024.2	(745 aa)	ATP-binding cassette sub-family D member 1 isoform 1 [Homo sapiens]
1:	NP_001427676.1	(845 aa)	ATP-binding cassette sub-family D member 1 isoform 2 [Homo sapiens]
2:	XP_047297873.1	(575 aa)	ATP-binding cassette sub-family D member 1 isoform X2 [Homo sapiens]


In [23]:
hum_sequences[0]

Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST')

# *Macaca fascicularis* sequences

There are 5 unique *Macaca fascicularis* ABCD1 protein sequences in the `fasta` file.

In [9]:
seq_ids = ['XP_065394918.1', 'XP_005594970.1', 'XP_065394919.1', 'XP_065394920.1', 'XP_073885705.1']

In [10]:
# Collect unique sequences from the species Macaca fascicularis
print('{0} orthologs: all Macaca fascicularis sequences\n'.format(gene))
all_sequences = []
seq_records = []
inc = 0
exc = 0
with Entrez.efetch(
    db="protein", rettype="gb", retmode="text", id=seq_ids,
) as handle:
    for seq_record in SeqIO.parse(handle, "gb"):
        sp = get_species(seq_record)
        seq = seq_record.seq
        if seq in all_sequences:
            print("\t{0}\t({1} aa)\t{2}\t*** the same as sequence {3} (excluded) ***".format(seq_record.id, len(seq), seq_record.description, all_sequences.index(seq)))
            exc += 1
        else:
            all_sequences.append(seq)
            print("{3}:\t{0}\t({1} aa)\t{2}".format(seq_record.id, len(seq), seq_record.description, inc))
            seq_records.append(seq_record)
            inc += 1

ABCD1 orthologs: all Macaca fascicularis sequences

0:	XP_065394918.1	(757 aa)	ATP-binding cassette sub-family D member 1 isoform X1 [Macaca fascicularis]
1:	XP_005594970.1	(745 aa)	ATP-binding cassette sub-family D member 1 isoform X2 [Macaca fascicularis]
2:	XP_065394919.1	(734 aa)	ATP-binding cassette sub-family D member 1 isoform X3 [Macaca fascicularis]
3:	XP_065394920.1	(592 aa)	ATP-binding cassette sub-family D member 1 isoform X4 [Macaca fascicularis]
4:	XP_073885705.1	(517 aa)	ATP-binding cassette sub-family D member 1 isoform X5 [Macaca fascicularis]


In [24]:
all_sequences

[Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AGL'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AST'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...WPM'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...IQT'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...TLH')]

# Finding the most similar *Macaca fascicularis* sequence to the reference human sequence

The following defines a pairwise aligner object using the Needleman-Wunsch
algorithm using the default `blastp` scoring scheme and substitution matrix.

In [13]:
from Bio import SeqIO, Align
from Bio.Seq import Seq

In [15]:
aligner = Align.PairwiseAligner(scoring="blastp")
print(aligner)

Pairwise sequence aligner with parameters
  substitution_matrix: <Array object at 0x143d08450>
  target_internal_open_gap_score: -12.000000
  target_internal_extend_gap_score: -1.000000
  target_left_open_gap_score: -12.000000
  target_left_extend_gap_score: -1.000000
  target_right_open_gap_score: -12.000000
  target_right_extend_gap_score: -1.000000
  query_internal_open_gap_score: -12.000000
  query_internal_extend_gap_score: -1.000000
  query_left_open_gap_score: -12.000000
  query_left_extend_gap_score: -1.000000
  query_right_open_gap_score: -12.000000
  query_right_extend_gap_score: -1.000000
  mode: global



In [16]:
print(aligner.substitution_matrix[:, :])

     A    B    C    D    E    F    G    H    I    J    K    L    M    N    O    P    Q    R    S    T    U    V    W    X    Y    Z    *
A  4.0 -2.0  0.0 -2.0 -1.0 -2.0  0.0 -2.0 -1.0 -1.0 -1.0 -1.0 -1.0 -2.0 -1.0 -1.0 -1.0 -1.0  1.0  0.0  0.0  0.0 -3.0 -1.0 -2.0 -1.0 -4.0
B -2.0  4.0 -3.0  4.0  1.0 -3.0 -1.0  0.0 -3.0 -3.0  0.0 -4.0 -3.0  4.0 -1.0 -2.0  0.0 -1.0  0.0 -1.0 -3.0 -3.0 -4.0 -1.0 -3.0  0.0 -4.0
C  0.0 -3.0  9.0 -3.0 -4.0 -2.0 -3.0 -3.0 -1.0 -1.0 -3.0 -1.0 -1.0 -3.0 -1.0 -3.0 -3.0 -3.0 -1.0 -1.0  9.0 -1.0 -2.0 -1.0 -2.0 -3.0 -4.0
D -2.0  4.0 -3.0  6.0  2.0 -3.0 -1.0 -1.0 -3.0 -3.0 -1.0 -4.0 -3.0  1.0 -1.0 -1.0  0.0 -2.0  0.0 -1.0 -3.0 -3.0 -4.0 -1.0 -3.0  1.0 -4.0
E -1.0  1.0 -4.0  2.0  5.0 -3.0 -2.0  0.0 -3.0 -3.0  1.0 -3.0 -2.0  0.0 -1.0 -1.0  2.0  0.0  0.0 -1.0 -4.0 -2.0 -3.0 -1.0 -2.0  4.0 -4.0
F -2.0 -3.0 -2.0 -3.0 -3.0  6.0 -3.0 -1.0  0.0  0.0 -3.0  0.0  0.0 -3.0 -1.0 -4.0 -3.0 -3.0 -2.0 -2.0 -2.0 -1.0  1.0 -1.0  3.0 -3.0 -4.0
G  0.0 -1.0 -3.0 -1.0 -2.0 -3.0  6.0 -2.0

The following shows that the second sequence is the most similar to the human
sequence.  It makes sense because it is the only one that has the same length.

In [31]:
max_score = aligner.score(hum_sequences[0], hum_sequences[0])
for i in range(5):
    score = aligner.score(hum_sequences[0], all_sequences[i])
    print(f"{seq_ids[i]}: {score/max_score:.4f}")

XP_065394918.1: 0.9515
XP_005594970.1: 0.9795
XP_065394919.1: 0.9538
XP_065394920.1: 0.6022
XP_073885705.1: 0.5793
